In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.DataFrame({
    'Name':   ['Asha', 'Ravi', 'Meena', 'Karan', 'Priya', 'Arjun', 'Neha'],
    'Course': ['CS', 'MBA', 'CS', 'MBA', 'CS', 'MBA', 'CS'],
    'City':   ['Bangalore', 'Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Delhi', 'Mumbai'],
    'Marks':  [88, 92, 79, 95, 84, 78, 91],
    'Age':    [22, 25, 23, 28, 24, 26, 22]
})

In [8]:
print(df)
print("\nShape:", df.shape)

    Name Course       City  Marks  Age
0   Asha     CS  Bangalore     88   22
1   Ravi    MBA     Mumbai     92   25
2  Meena     CS      Delhi     79   23
3  Karan    MBA  Bangalore     95   28
4  Priya     CS    Chennai     84   24
5  Arjun    MBA      Delhi     78   26
6   Neha     CS     Mumbai     91   22

Shape: (7, 5)


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Name    7 non-null      object
 1   Course  7 non-null      object
 2   City    7 non-null      object
 3   Marks   7 non-null      int64 
 4   Age     7 non-null      int64 
dtypes: int64(2), object(3)
memory usage: 412.0+ bytes


In [13]:
df.describe()

,Marks,Age
count,7.000000,7.000000
mean,86.714286,24.285714
std,6.575568,2.214670
min,78.000000,22.000000
25%,81.500000,22.500000
50%,88.000000,24.000000
75%,91.500000,25.500000
max,95.000000,28.000000


In [22]:
#Basic group by
print(f"Average marks of students: {df['Marks'].mean}")
print("\n",df.groupby('Course')['Marks'].mean)

print(f'Multiple Aggregations on one column:')
print(df.groupby('Course')['Marks'].agg(['mean', 'median', 'max', 'min', 'count']))

print('Multiple columns aggregated:')
print(df.groupby('Course')[['Marks', 'Age']].mean())

print('Group by multiple columns:')
print(df.groupby(['Course', 'City'])[['Marks', 'Age']].mean())

print('\nAggregate with different function')
print(df.groupby('Course').agg({
    'Marks': ['mean', 'median'],
    'Age': ['min', 'max']
}))

#transform return same length series
df['course_avg_marks'] = df.groupby('Course')['Marks'].transform('mean')
print('\nDataFrame with group average needed:\n',df)

#filter: keeps group meeting a condition
high_avg = df.groupby('Course').filter(lambda mark: mark['Marks'].mean() > 85.9)
print("\nCourses with average marks > 85.9:\n", high_avg)

Average marks of students: <bound method Series.mean of 0    88
1    92
2    79
3    95
4    84
5    78
6    91
Name: Marks, dtype: int64>

 <bound method GroupBy.mean of <pandas.core.groupby.generic.SeriesGroupBy object at 0x7cc54de07a70>>
Multiple Aggregations on one column:
             mean  median  max  min  count
Course                                    
CS      85.500000    86.0   91   79      4
MBA     88.333333    92.0   95   78      3
Multiple columns aggregated:
            Marks        Age
Course                      
CS      85.500000  22.750000
MBA     88.333333  26.333333
Group by multiple columns:
                  Marks   Age
Course City                  
CS     Bangalore   88.0  22.0
       Chennai     84.0  24.0
       Delhi       79.0  23.0
       Mumbai      91.0  22.0
MBA    Bangalore   95.0  28.0
       Delhi       78.0  26.0
       Mumbai      92.0  25.0

Aggregate with different function
            Marks        Age    
             mean median min max
Course 

# Merge / Join

In [30]:
#merge is like SQL JOIN
students = pd.DataFrame({
    'StudentID': [1, 2, 3, 4, 5],
    'Name':      ['Asha', 'Ravi', 'Meena', 'Karan', 'Priya'],
    'CourseID':  [101, 102, 101, 103, 102]
})

courses = pd.DataFrame({
    'CourseID':   [101, 102, 103, 104],
    'CourseName': ['CS', 'MBA', 'Design', 'Marketing']
})

scores = pd.DataFrame({
    'StudentID': [1, 2, 3, 5, 6],   # note: 4 missing, 6 doesn't exist in students
    'Score':     [88, 92, 79, 84, 95]
})

#INNER JOIN: only matching rows from both
inner_join = pd.merge(students, scores, on='StudentID', how='inner')
print('INNER JOIN (students + courses):\n', inner_join)

#LEFT JOIN: all rows from left
left_join = pd.merge(students, scores, on='StudentID', how='left')
print("\nLEFT JOIN:\n", left_join)

#RIGHT JOIN:all row from right
right_join = pd.merge(students, scores, on='StudentID', how='right')
print("\nRIGHT JOIN:\n", right_join)

#OUTER JOIN: all rows from both
outer_join = pd.merge(students, scores, on='StudentID', how='outer')
print("\nOUTER JOIN:\n", outer_join)

#Chaining merges
full = students.merge(courses, on='CourseID').merge(scores, on='StudentID', how='left')
print('\nFull merged table:\n',full)

#join on different column names(outer):
df1 = pd.DataFrame({'ID': [1, 2, 3], 'Val': ['a', 'b', 'c']})
df2 = pd.DataFrame({'Num': [1, 2, 4], 'Info': ['x', 'y', 'z']})
print('\nMerge on different column names:\n',
      pd.merge(df1, df2, left_on='ID', right_on='Num', how='outer'))

#join on different column names (inner):
print('\nMerge on different column names:\n',
      pd.merge(df1, df2, left_on='ID', right_on='Num', how='inner'))

INNER JOIN (students + courses):
    StudentID   Name  CourseID  Score
0          1   Asha       101     88
1          2   Ravi       102     92
2          3  Meena       101     79
3          5  Priya       102     84

LEFT JOIN:
    StudentID   Name  CourseID  Score
0          1   Asha       101   88.0
1          2   Ravi       102   92.0
2          3  Meena       101   79.0
3          4  Karan       103    NaN
4          5  Priya       102   84.0

RIGHT JOIN:
    StudentID   Name  CourseID  Score
0          1   Asha     101.0     88
1          2   Ravi     102.0     92
2          3  Meena     101.0     79
3          5  Priya     102.0     84
4          6    NaN       NaN     95

OUTER JOIN:
    StudentID   Name  CourseID  Score
0          1   Asha     101.0   88.0
1          2   Ravi     102.0   92.0
2          3  Meena     101.0   79.0
3          4  Karan     103.0    NaN
4          5  Priya     102.0   84.0
5          6    NaN       NaN   95.0

Full merged table:
    StudentID   N

# APPLY / MAP

In [32]:
df = pd.DataFrame({
    'Name':   ['Asha', 'Ravi', 'Meena', 'Karan', 'Priya'],
    'Marks':  [88, 92, 79, 95, 84],
    'Course': ['cs', 'mba', 'CS', 'Mba', 'cs']   # messy casing
})

# --- map: element-wise on a Series ---
grade_map = {
    range(90, 101): 'A',
    range(80, 90):  'B',
    range(70, 80):  'C'
}

def assign_grade(mark):
    if mark >= 90: return 'A'
    elif mark >= 80: return 'B'
    else: return 'C'

df['Grade'] = df['Marks'].map(assign_grade)
print("With Grades:\n", df)

# map on string column (simple value replacement)
df['Course_Clean'] = df['Course'].str.upper()
print("\nCleaned Course:\n", df[['Course', 'Course_Clean']])

# --- apply: row-wise or column-wise on DataFrame ---
# axis=1 → apply function to each ROW
def student_summary(row):
    return f"{row['Name']} scored {row['Marks']} ({row['Grade']})"

df['Summary'] = df.apply(student_summary, axis=1)
print("\nSummary column:\n", df['Summary'])

# axis=0 → apply function to each COLUMN
print("\nColumn-wise apply (mean of numeric cols):")
print(df[['Marks']].apply(np.mean, axis=0))

# --- applymap (now called map in newer Pandas) for element-wise on full DataFrame ---
df_nums = pd.DataFrame({'A': [1.123, 2.456], 'B': [3.789, 4.012]})
print("\nRounded DataFrame:")
print(df_nums.apply(lambda x: x.round(2)))

# --- lambda shorthand ---
df['Marks_Scaled'] = df['Marks'].apply(lambda x: (x - 70) / 30)
print("\nScaled Marks (manual min-max):\n", df[['Name', 'Marks', 'Marks_Scaled']])

With Grades:
     Name  Marks Course Grade
0   Asha     88     cs     B
1   Ravi     92    mba     A
2  Meena     79     CS     C
3  Karan     95    Mba     A
4  Priya     84     cs     B

Cleaned Course:
   Course Course_Clean
0     cs           CS
1    mba          MBA
2     CS           CS
3    Mba          MBA
4     cs           CS

Summary column:
 0     Asha scored 88 (B)
1     Ravi scored 92 (A)
2    Meena scored 79 (C)
3    Karan scored 95 (A)
4    Priya scored 84 (B)
Name: Summary, dtype: object

Column-wise apply (mean of numeric cols):
Marks    87.6
dtype: float64

Rounded DataFrame:
      A     B
0  1.12  3.79
1  2.46  4.01

Scaled Marks (manual min-max):
     Name  Marks  Marks_Scaled
0   Asha     88      0.600000
1   Ravi     92      0.733333
2  Meena     79      0.300000
3  Karan     95      0.833333
4  Priya     84      0.466667


# Sorting & Handling Duplicates

In [33]:
df = pd.DataFrame({
    'Name':   ['Asha', 'Ravi', 'Meena', 'Karan', 'Priya', 'Asha', 'Ravi'],
    'Course': ['CS', 'MBA', 'CS', 'MBA', 'CS', 'CS', 'MBA'],
    'Marks':  [88, 92, 79, 95, 84, 88, 91],
    'City':   ['Bangalore', 'Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Bangalore', 'Pune']
})

print("Original DataFrame:\n", df)

# --- Sorting ---
print("\nSort by Marks (descending):")
print(df.sort_values('Marks', ascending=False))

print("\nSort by Course (asc) then Marks (desc):")
print(df.sort_values(['Course', 'Marks'], ascending=[True, False]))

print("\nSort by index:")
print(df.sort_index(ascending=False))

# --- Duplicates ---
print("\nDuplicate rows:")
print(df[df.duplicated()])                        # rows that are exact duplicates

print("\nDuplicate by Name only:")
print(df[df.duplicated(subset=['Name'])])          # duplicates based on Name column

print("\nDuplicate check (True/False for each row):")
print(df.duplicated())

# --- Dropping duplicates ---
df_no_dup = df.drop_duplicates()
print("\nAfter drop_duplicates (exact):\n", df_no_dup)

df_keep_last = df.drop_duplicates(subset=['Name'], keep='last')
print("\nKeep last occurrence per Name:\n", df_keep_last)

df_keep_first = df.drop_duplicates(subset=['Name'], keep='first')
print("\nKeep first occurrence per Name:\n", df_keep_first)

Original DataFrame:
     Name Course  Marks       City
0   Asha     CS     88  Bangalore
1   Ravi    MBA     92     Mumbai
2  Meena     CS     79      Delhi
3  Karan    MBA     95  Bangalore
4  Priya     CS     84    Chennai
5   Asha     CS     88  Bangalore
6   Ravi    MBA     91       Pune

Sort by Marks (descending):
    Name Course  Marks       City
3  Karan    MBA     95  Bangalore
1   Ravi    MBA     92     Mumbai
6   Ravi    MBA     91       Pune
5   Asha     CS     88  Bangalore
0   Asha     CS     88  Bangalore
4  Priya     CS     84    Chennai
2  Meena     CS     79      Delhi

Sort by Course (asc) then Marks (desc):
    Name Course  Marks       City
0   Asha     CS     88  Bangalore
5   Asha     CS     88  Bangalore
4  Priya     CS     84    Chennai
2  Meena     CS     79      Delhi
3  Karan    MBA     95  Bangalore
1   Ravi    MBA     92     Mumbai
6   Ravi    MBA     91       Pune

Sort by index:
    Name Course  Marks       City
6   Ravi    MBA     91       Pune
5   Asha 